In [1]:
import pandas as pd
import numpy as np
import json
import re
from pathlib import Path
import seaborn as sns
from matplotlib import pyplot as plt

In [2]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

## Collect runs

In [3]:
# All run folders to include, relative to project root.
# Each entry is (folder_path, folder_tag) – tag appears in the DataFrame.
# PROJECT_ROOT = Path("/cephfs/home/bulatov/2026/autoresearch/compressing-associations-gdn")
PROJECT_ROOT = Path("/home/bulatov/rmt/test-time/compressing-associations-gdn")
RUNS_FOLDERS = [
    # (PROJECT_ROOT / "runs-rmmv5p4", "rmmv5p4"),
    # (PROJECT_ROOT / "runs-rmmv5p7", "rmmv5p7"),
    # (PROJECT_ROOT / "runs-rmmv5p7p1", "rmmv5p7p1"),
    # (PROJECT_ROOT / "runs-rmmv6p2", "rmmv6p2"),
    # (PROJECT_ROOT / "runs-rmmv6p3", "rmmv6p3"),
    # (PROJECT_ROOT / "runs-rmmv6p4", "rmmv6p4"),
    # (PROJECT_ROOT / "runs-rmmv6p4-armt", "rmmv6p4-armt"),
    # (PROJECT_ROOT / "runs-rmmv6p5", "rmmv6p5"),
    
    # (PROJECT_ROOT / "runs-baselines","segment_baselines"),
    # (PROJECT_ROOT / "runs-conv-study","conv_study"),
    (PROJECT_ROOT / "runs-rebuttal","rebuttal"),


    # (Path("/home/bulatov/rmt/test-time/compressing-associations/runs-rebuttal-default"), "recurrent_baselines"),
    # (Path("/home/bulatov/rmt/test-time/compressing-associations-gdn/runs-rebuttal-default"), "recurrent_baselines"),
]


def get_eval_from_trainer_state(run_path):
    """Fallback: read best-checkpoint eval metrics from trainer_state.json."""
    # Some runs have trainer_state.json directly in run_path (newer format)
    ts_path = run_path / 'trainer_state.json'
    if not ts_path.exists():
        # Older format: trainer_state lives inside the checkpoint subdirectory
        checkpoints = sorted(
            run_path.glob('checkpoint-*/trainer_state.json'),
            key=lambda p: int(p.parent.name.split('-')[1])
        )
        if checkpoints:
            ts_path = checkpoints[-1]
    if not ts_path.exists():
        return {}

    ts = json.load(open(ts_path))

    # Determine the best checkpoint step
    best_cpt = ts.get('best_model_checkpoint', '')
    m = re.search(r'checkpoint-(\d+)', best_cpt or '')
    best_step = int(m.group(1)) if m else None

    # Collect all log entries that have eval metrics
    eval_entries = [e for e in ts.get('log_history', []) if 'eval_exact_match' in e]
    if not eval_entries:
        return {}

    if best_step is not None:
        matching = [e for e in eval_entries if e.get('step') == best_step]
        entry = matching[0] if matching else eval_entries[-1]
    else:
        entry = eval_entries[-1]

    return {k: v for k, v in entry.items() if k.startswith('eval_')}


all_runs = []

for runs_path, folder_tag in RUNS_FOLDERS:
    if not runs_path.exists():
        print(f"Missing: {runs_path}")
        continue

    for run_path in sorted(runs_path.glob('**/run_*')):
        if not run_path.is_dir():
            continue
        if 'checkpoint' in str(run_path):
            continue

        stats = {}
        stats['folder_tag']  = folder_tag
        stats['run_path']    = str(run_path)
        stats['run_name']    = run_path.parent.name       # e.g. rmmv5_GatedDeltaNet_...
        stats['task_name']   = run_path.parent.parent.name  # e.g. N4-K2V2-V62_1M
        stats['run_id']      = int(run_path.name.split('_')[1])

        config_path = run_path / 'config.json'
        if config_path.exists():
            cli_args = json.load(open(config_path))['cli_args']
            stats.update(cli_args)

        results_path = run_path / 'all_results.json'
        if results_path.exists():
            results = json.load(open(results_path))
            stats.update(results)
        else:
            # Fallback: recover eval metrics from trainer_state.json
            fallback = get_eval_from_trainer_state(run_path)
            if fallback:
                stats.update(fallback)
                stats.setdefault('eval_exact_match',    np.nan)
                stats.setdefault('eval_token_accuracy', np.nan)
            else:
                stats['eval_exact_match']    = np.nan
                stats['eval_token_accuracy'] = np.nan

        all_runs.append(stats)

df = pd.DataFrame(all_runs)
print(f"Total runs: {len(df)}")
print(f"Columns: {list(df.columns)}")


Total runs: 69
Columns: ['folder_tag', 'run_path', 'run_name', 'task_name', 'run_id', 'exp_path', 'per_device_batch_size', 'data_path', 'tokenizer_path', 'gradient_accumulation_steps', 'total_batch_size', 'metric_for_best_model', 'warmup_steps', 'max_steps', 'logging_steps', 'eval_steps', 'weight_decay', 'learning_rate', 'lr_scheduler_type', 'early_stopping_patience', 'seed', 'base_model', 'n_layer', 'n_head', 'n_embd', 'fla_layer', 'state_size', 'expand_v', 'conv_kernel', 'use_short_conv', 'num_memory_vectors', 'write_mode', 'read_mode', 'write_value_dim', 'num_memory_heads', 'use_parallel_prefill', 'memory_task_freq', 'memory_task', 'memory_key_size', 'memory_value_size', 'model_cpt', 'checkpoint', 'tokens_per_segment', 'n_pairs', 'n_keys', 'n_values', 'epoch', 'eval_exact_match', 'eval_exact_match_None', 'eval_loss', 'eval_runtime', 'eval_samples_per_second', 'eval_steps_per_second', 'eval_token_accuracy', 'patience']


In [4]:
# df.eval_exact_match = df.eval_exact_match.fillna(df.eval_exact_match)

In [5]:
df.folder_tag.value_counts()

folder_tag
rebuttal    69
Name: count, dtype: int64

In [6]:
df.max_steps.value_counts()


max_steps
200000    69
Name: count, dtype: int64

In [7]:
# def get_final_step(run_path: Path) -> float:
#     """Last global_step from trainer_state (run dir or latest checkpoint)."""
#     run_path = Path(run_path)
#     ts_path = run_path / "trainer_state.json"
#     if not ts_path.exists():
#         checkpoints = sorted(
#             run_path.glob("checkpoint-*/trainer_state.json"),
#             key=lambda p: int(p.parent.name.split("-")[1]),
#         )
#         if not checkpoints:
#             return np.nan
#         ts_path = checkpoints[-1]
#     ts = json.load(open(ts_path))
#     return ts.get("global_step", np.nan)
# df["final_step"] = df["run_path"].apply(lambda p: get_final_step(Path(p)))
# df["completed"] = df["final_step"] >= df["max_steps"]
# EM_THRESH = 0.90  # eval_exact_match is 0–1 in df, not percent
# df = df[
#     df["completed"]
#     | (df["eval_exact_match"] >= EM_THRESH)
# ].copy()

# # df["has_all_results"] = df["run_path"].apply(
# #     lambda p: (Path(p) / "all_results.json").exists()
# # )

# # # strict: only runs that finished the script
# # df_finished = df[df["has_all_results"]]

# # # or combine with your 90% rule for fallback-only high performers
# # df_ok = df[
# #     df["has_all_results"]
# #     | (df["eval_exact_match"] >= 0.90)  # rescue intentional / lucky early peaks
# # ]

# # # if you add best_step when collecting (from checkpoint-N in best_model_checkpoint)
# # df_ok = df[
# #     (df["final_step"] >= df["max_steps"])
# #     | (df["eval_exact_match"] >= 0.90)
# # ]

# df.shape

## Parse / normalise columns

In [8]:
# ── Parse version tag and a few fields not guaranteed to be in cli_args ─────

def parse_run_name(run_name: str) -> dict:
    """
    Extract structured parameters from a run-name string.

    Run-name patterns
    -----------------
    rmmv5 / rmmv5p / rmmv5p1:
      {version}_{FLA}_{base}_L{L}H{H}D{D}
          _ss{ss}_M{M}_{read}[H{nmh}]_{write}[-md{wvd}|-wvd{wvd}]
          _lr{lr}_bs{bs}[_pps{pps}][_tps{tps}]

    rmmv3:
      {version}_{FLA}_{base}_L{L}H{H}D{D}
          _ss{ss}_M{M}_{mode}     <- single mode = both read & write
          _lr{lr}_bs{bs}_pps{pps}

    segment_baselines (armt / rmt):
      {version}_{base}_L{L}H{H}D{D}_mem{nmem}[d{dmem}]_lr{lr}[-{nc}x{np}]_bs{bs}[-gen]

    recurrent_baselines (gated_delta_net / mamba / mamba2):
      {model}_L{L}D{D}_ss{ss}[_ck{ck}|_noconv]_bs_{bs}_lr_{lr}_b2_{b2}
    """
    info = {}

    # Version / model family
    # Order matters: mamba2 before mamba, gated_delta_net before others
    m = re.match(r'^(rmmv\d+(?:p\d*)?|armt|rmt|gated_delta_net|mamba2?)', run_name)
    if m:
        info['version'] = m.group(1)

    # Architecture dimensions: L, H, D  (H is optional in recurrent_baselines)
    m = re.search(r'_L(\d+)H(\d+)D(\d+)', run_name)
    if m:
        info['L'] = int(m.group(1))
        info['H'] = int(m.group(2))
        info['D'] = int(m.group(3))
    else:
        # Fallback: L{L}D{D} without H (recurrent_baselines naming)
        m = re.search(r'_L(\d+)D(\d+)', run_name)
        if m:
            info['L'] = int(m.group(1))
            info['D'] = int(m.group(2))

    # state_size  (_ss{N})
    m = re.search(r'_ss(\d+)', run_name)
    if m:
        info['ss_parsed'] = int(m.group(1))

    # num_memory_vectors  (_M{N}_)
    m = re.search(r'_M(\d+)_', run_name)
    if m:
        info['M_parsed'] = int(m.group(1))

    # n_mem_tokens for segment_baselines  (_mem{N})
    m = re.search(r'_mem(\d+)', run_name)
    if m:
        info['mem_parsed'] = int(m.group(1))

    # write_value_dim  (-md{N}  or  -wvd{N})
    m = re.search(r'-(?:md|wvd)(\d+)', run_name)
    if m:
        info['write_value_dim_parsed'] = int(m.group(1))

    # conv_kernel  (_ck{N}) – present in recurrent_baselines and rmmv5p1
    m = re.search(r'_ck(\d+)', run_name)
    if m:
        info['ck_parsed'] = int(m.group(1))

    # pairs_per_segment (_pps{N})
    m = re.search(r'_pps(\d+)', run_name)
    if m:
        info['pps_parsed'] = int(m.group(1))

    # tokens_per_segment (_tps{N})
    m = re.search(r'_tps(\d+)', run_name)
    if m:
        info['tps_parsed'] = int(m.group(1))

    # Read/write mode from name (fallback for runs where config.json is absent)
    # Known modes: unpool, pool, cross_attn, identity
    _M = r'(unpool|pool|cross_attn|identity)'

    # Two-mode: _M{N}_{read}[H\d+]_{write}[-md/wvd{N}] followed by _ev/_ck/_lr/_bs
    two = re.search(
        r'_M\d+_' + _M + r'(?:H\d+)?_' + _M + r'(?:-(?:md|wvd)\d+)?(?=_(?:ev|ck|lr|bs))',
        run_name
    )
    if two:
        info['read_mode_parsed']  = two.group(1)
        info['write_mode_parsed'] = two.group(2)
    else:
        # Single-mode (rmmv3): _M{N}_{mode}_lr
        one = re.search(r'_M\d+_' + _M + r'_lr', run_name)
        if one:
            info['read_mode_parsed']  = one.group(1)
            info['write_mode_parsed'] = one.group(1)

    return info


parsed = df['run_name'].apply(parse_run_name).apply(pd.Series)
df = pd.concat([df, parsed], axis=1)


In [9]:
# ── Unified column names across all model families ────────────────────────

# state_size: prefer cli_args value; fall back to parsed
if 'state_size' not in df.columns:
    df['state_size'] = np.nan
df['state_size'] = df['state_size'].fillna(df.get('ss_parsed'))

# num_memory_vectors: cli_args > M_parsed; segment_baselines use n_mem_tokens
if 'num_memory_vectors' not in df.columns:
    df['num_memory_vectors'] = np.nan
df['num_memory_vectors'] = df['num_memory_vectors'].fillna(df.get('M_parsed'))
if 'n_mem_tokens' in df.columns:
    df['num_memory_vectors'] = df['num_memory_vectors'].fillna(df['n_mem_tokens'])

# write_value_dim: prefer cli_args; fall back to name-parsed
if 'write_value_dim' not in df.columns:
    df['write_value_dim'] = np.nan
df['write_value_dim'] = df['write_value_dim'].fillna(df.get('write_value_dim_parsed'))

# read_mode / write_mode: fill NaN with name-parsed, then 'N/A' for baselines
for col in ['read_mode', 'write_mode', 'fla_layer']:
    if col not in df.columns:
        df[col] = np.nan

# Prefer cli_args value; fall back to what was parsed from the run name
if 'read_mode_parsed' in df.columns:
    df['read_mode'] = df['read_mode'].fillna(df['read_mode_parsed'])
if 'write_mode_parsed' in df.columns:
    df['write_mode'] = df['write_mode'].fillna(df['write_mode_parsed'])

# rmmv3 uses a single mode for both read and write; only write_mode is in config
rmmv3_mask = (
    (df['folder_tag'] == 'rmmv3')
    & df['read_mode'].isna()
    & df['write_mode'].notna()
)
df.loc[rmmv3_mask, 'read_mode'] = df.loc[rmmv3_mask, 'write_mode']

# Final fill: 'N/A' for any remaining NaN
for col in ['read_mode', 'write_mode', 'fla_layer']:
    df[col] = df[col].fillna('N/A')

# tokens_per_segment: prefer cli_args
if 'tokens_per_segment' not in df.columns:
    df['tokens_per_segment'] = np.nan
df['tokens_per_segment'] = df['tokens_per_segment'].fillna(df.get('tps_parsed'))

# pairs_per_segment: prefer cli_args
if 'pairs_per_segment' not in df.columns:
    df['pairs_per_segment'] = np.nan
df['pairs_per_segment'] = df['pairs_per_segment'].fillna(df.get('pps_parsed'))

# n_pairs from task_name
df['n_pairs_task'] = df['task_name'].str.extract(r'^N(\d+)-').astype(float)

# Convenience label: write_mode / write_value_dim combined
df['write_config'] = df['write_mode'].astype(str)
mask_md = df['write_value_dim'].notna()
df.loc[mask_md, 'write_config'] = (
    df.loc[mask_md, 'write_mode'].astype(str) + '-md' +
    df.loc[mask_md, 'write_value_dim'].astype(int).astype(str)
)

# Short label for legend: folder_tag + read_mode + write_config
df['config_label'] = (
    df['folder_tag'] + '|' +
    df['read_mode'].astype(str) + '→' +
    df['write_config'].astype(str)
)

df.shape


(69, 69)

In [10]:
df.loc[df.use_short_conv == False, 'conv_kernel'] = 0

In [11]:
def get_pps_from_tps(tps):
    if tps == 32:
        return 4
    else:
        return round(tps/7)

df.loc[df.pairs_per_segment == 0, 'pairs_per_segment'] = df.loc[df.pairs_per_segment == 0, 'tokens_per_segment'].apply(get_pps_from_tps)

## Groupby table

In [12]:
# Effective conv kernel: 0 = no conv, else the actual kernel size
df['conv_type'] = df.apply(
    lambda r: 'noconv' if r.get('no_conv') is True
            else (f"ck{int(r['conv_kernel'])}" if pd.notna(r.get('conv_kernel')) else 'N/A'),
    axis=1
)

In [13]:
for col in ["num_memory_vectors", "pairs_per_segment"]:
    df[col] = df[col].fillna(-1).astype(int)

In [14]:
df_valid = df.dropna(subset=['eval_exact_match']).copy()

# df_valid = df_valid[df_valid['write_config'] == 'identity']

GROUP_COLS = [
    'task_name',
    'folder_tag',
    'base_model',
    'n_layer',
    'n_head',
    'n_embd',
    'state_size',
    # 'conv_kernel',
    'conv_type',
    'num_memory_vectors',
    'read_mode',
    'write_config',
    # 'tokens_per_segment',
    'pairs_per_segment',
    'learning_rate',
]

# Drop group cols that are entirely NaN / not in df
GROUP_COLS = [c for c in GROUP_COLS if c in df_valid.columns and df_valid[c].notna().any()]

grouped = df_valid.groupby(GROUP_COLS, dropna=False).agg(
    avg_em  = ('eval_exact_match', lambda x: round(np.nanmean(x) * 100, 2)),
    std_em  = ('eval_exact_match', lambda x: round(np.nanstd(x) * 100, 2)),
    all_ems = ('eval_exact_match', lambda x: tuple(round(v * 100, 2) for v in x if not np.isnan(v))),
    n_runs  = ('eval_exact_match', 'count'),
)

# Sort by task n_pairs then avg_em descending
gr = grouped.copy()
gr['_n_pairs_sort'] = gr.index.get_level_values('task_name').str.extract(r'^N(\d+)-').astype(float).values
gr = gr.sort_values(['_n_pairs_sort', 'avg_em'], ascending=[True, False]).drop(columns='_n_pairs_sort')
# gr already has GROUP_COLS as index; no need to set_index again
# grouped # is large

In [15]:
grouped

avg_em  \
task_name      folder_tag base_model n_layer n_head n_embd state_size conv_type num_memory_vectors read_mode  write_config pairs_per_segment learning_rate           
N8-K2V2-V62_1M rebuttal   llama      4       4      128    32         ck4       32                 cross_attn cross_attn   1                 0.00001          6.72   
                                                                                                                                             0.00005         62.50   
                                                                                                                                             0.00010         93.50   
                                                                                                                                             0.00030         65.81   
                                                                                                                                             0.00050          9.68   
                                                                                                                                             0.00070          4.75   
                                                                                                                                             0.00100          3.85   
                                                                                                                                             0.00300          0.04   
                                                                                                                                             0.00500          0.04   
                                                                                                   identity   identity     8                 0.00001         13.01   
                                                                                                                                             0.00005         51.99   
                                                                                                                                             0.00010         55.83   
                                                                                                                                             0.00030         95.67   
                                                                                                                                             0.00050         95.68   
                                                                                                                                             0.00070         97.96   
                                                                                                                                             0.00100         47.01   
                                                                                                                                             0.00300         13.45   
                                                                                                                                             0.00500         84.24   
                                                                                                   unpool     pool         1                 0.00001         28.53   
                                                                                                                                             0.00005         98.77   
                                                                                                                                             0.00010         70.01   
                                                                                                                                             0.00030         56.71   
                                                                                                                                             0.00050         13.17   
      

In [21]:
g

,task_name,folder_tag,base_model,n_layer,n_head,n_embd,state_size,conv_type,num_memory_vectors,read_mode,write_config,pairs_per_segment,learning_rate,avg_em,std_em,all_ems,n_runs,model,lr
0,N8-K2V2-V62_1M,rebuttal,llama,4,4,128,32,ck4,32,cross_attn,cross_attn,1,0.00001,6.72,3.77,"(3.74, 4.38, 12.04)",3,segment-lvl cross-attn,0.00001
1,N8-K2V2-V62_1M,rebuttal,llama,4,4,128,32,ck4,32,cross_attn,cross_attn,1,0.00005,62.50,35.39,"(94.88, 13.26, 79.36)",3,segment-lvl cross-attn,0.00005
2,N8-K2V2-V62_1M,rebuttal,llama,4,4,128,32,ck4,32,cross_attn,cross_attn,1,0.00010,93.50,4.57,"(87.82, 99.0, 93.68)",3,segment-lvl cross-attn,0.0001
3,N8-K2V2-V62_1M,rebuttal,llama,4,4,128,32,ck4,32,cross_attn,cross_attn,1,0.00030,65.81,37.39,"(92.06, 92.44, 12.94)",3,segment-lvl cross-attn,0.0003
4,N8-K2V2-V62_1M,rebuttal,llama,4,4,128,32,ck4,32,cross_attn,cross_attn,1,0.00050,9.68,2.48,"(6.38, 10.3, 12.36)",3,segment-lvl cross-attn,0.0005
5,N8-K2V2-V62_1M,rebuttal,llama,4,4,128,32,ck4,32,cross_attn,cross_attn,1,0.00070,4.75,1.33,"(6.62, 3.98, 3.64)",3,segment-lvl cross-attn,0.0007
6,N8-K2V2-V62_1M,rebuttal,llama,4,4,128,32,ck4,32,cross_attn,cross_attn,1,0.00100,3.85,5.37,"(11.44, 0.06, 0.04)",3,segment-lvl cross-attn,0.001
7,N8-K2V2-V62_1M,rebuttal,llama,4,4,128,32,ck4,32,cross_attn,cross_attn,1,0.00300,0.04,0.00,"(0.04, 0.04, 0.04)",3,segment-lvl cross-attn,0.003
8,N8-K2V2-V62_1M,rebuttal,llama,4,4,128,32,ck4,32,cross_attn,cross_attn,1,0.00500,0.04,0.00,"(0.04, 0.04)",2,segment-lvl cross-attn,0.005
9,N8-K2V2-V62_1M,rebuttal,llama,4,4,128,32,ck4,32,identity,identity,8,0.00001,13.01,0.15,"(13.16, 12.86)",2,token-lvl,0.00001


In [27]:
g = grouped.reset_index()
read_mode_to_model = {
    "identity": "token-lvl",
    "unpool": "segment-lvl pool",
    "cross_attn": "segment-lvl cross-attn",
}
best_read_mode_to_model = {
    "unpool": "segment-lvl pool (best)",
    "cross_attn": "segment-lvl cross-attn (best)",
}
main_model_order = list(read_mode_to_model.values())
best_model_order = list(best_read_mode_to_model.values())

def _fmt_lr(x):
    s = format(x, ".5f")
    return s.rstrip("0").rstrip(".") if "." in s else str(x)

learning_rates = np.sort(g["learning_rate"].unique())
lr_strs = [_fmt_lr(lr) for lr in learning_rates]
g["lr"] = pd.Categorical(g["learning_rate"].map(_fmt_lr), categories=lr_strs, ordered=True)
g["cell"] = g.apply(lambda r: f"{r['avg_em']:.1f} ± {r['std_em']:.1f}", axis=1)
g["max_cell"] = g["all_ems"].apply(lambda xs: f"{max(xs):.1f}" if xs else "")

# drop highest LR (same as previous plot)
g = g[g["lr"] != lr_strs[-1]]
lr_cols = lr_strs[:-1]

main_table = (
    g.assign(model=g["read_mode"].map(read_mode_to_model))
    .pivot_table(index="model", columns="lr", values="cell", aggfunc="first")
    .reindex(index=main_model_order, columns=lr_cols)
)
best_table = (
    g[g["read_mode"].isin(best_read_mode_to_model)]
    .assign(model=g["read_mode"].map(best_read_mode_to_model))
    .pivot_table(index="model", columns="lr", values="max_cell", aggfunc="first")
    .reindex(index=best_model_order, columns=lr_cols)
)

hline = pd.DataFrame([{col: "—" for col in lr_cols}], index=[""])
hline.index.name = "model"
table = pd.concat([main_table, hline, best_table])
table.columns.name = "lr"

display(
    table.style.apply(
        lambda row: ["border-top: 2px solid black" if row.name == "" else "" for _ in row],
        axis=1,
    )
)
print(table.to_markdown())


,0.00001,0.00005,0.0001,0.0003,0.0005,0.0007,0.001,0.003
model,,,,,,,,
token-lvl,13.0 ± 0.1,52.0 ± 38.4,55.8 ± 42.7,95.7 ± 0.8,95.7 ± 3.3,98.0 ± 0.1,47.0 ± 33.8,13.4 ± 0.0
segment-lvl pool,28.5 ± 22.1,98.8 ± 0.5,70.0 ± 40.3,56.7 ± 35.2,13.2 ± 0.2,62.0 ± 33.0,9.1 ± 6.9,0.0 ± 0.0
segment-lvl cross-attn,6.7 ± 3.8,62.5 ± 35.4,93.5 ± 4.6,65.8 ± 37.4,9.7 ± 2.5,4.8 ± 1.3,3.9 ± 5.4,0.0 ± 0.0
,,,,,,,,
segment-lvl pool (best),,98.8 ± 0.5,,,,,,
segment-lvl cross-attn (best),,,93.5 ± 4.6,,,,,


| model                  | 0.00001     | 0.00005     | 0.0001      | 0.0003      | 0.0005     | 0.0007      | 0.001       | 0.003      |
|:-----------------------|:------------|:------------|:------------|:------------|:-----------|:------------|:------------|:-----------|
| token-lvl              | 13.0 ± 0.1  | 52.0 ± 38.4 | 55.8 ± 42.7 | 95.7 ± 0.8  | 95.7 ± 3.3 | 98.0 ± 0.1  | 47.0 ± 33.8 | 13.4 ± 0.0 |
| segment-lvl pool       | 28.5 ± 22.1 | 98.8 ± 0.5  | 70.0 ± 40.3 | 56.7 ± 35.2 | 13.2 ± 0.2 | 62.0 ± 33.0 | 9.1 ± 6.9   | 0.0 ± 0.0  |
| segment-lvl cross-attn | 6.7 ± 3.8   | 62.5 ± 35.4 | 93.5 ± 4.6  | 65.8 ± 37.4 | 9.7 ± 2.5  | 4.8 ± 1.3   | 3.9 ± 5.4   | 0.0 ± 0.0  |

---

| model                         | 0.00001   | 0.00005    | 0.0001     | 0.0003   | 0.0005   | 0.0007   | 0.001   | 0.003   |
|:------------------------------|:----------|:-----------|:-----------|:---------|:---------|:---------|:--------|:--------|
| segment-lvl pool (best)       |           | 98.8 ± 0.5 | 